# V10 FraudGT · 공유그래프 실습형

**2026-10-27 조건 수정본. 기존 실습형은 거래별 개별 그래프 비교용으로 유지합니다.**

이 노트북은 `shared_corrected_001` 새 실행을 사용합니다. 기존 학습의 가중치·기록을 가져오지 않습니다. 두 노트북은 같은 V10 피처, 거래 순서, 기간 분할, 손실, 검증 조건을 사용하며 그래프 공유 방식이 다릅니다.

1. 이 파일을 다시 연 뒤 **커널을 재시작**합니다. 권장 커널: `/workspace/kimi/venv/bin/python`.
2. 첫 코드 셀에서 `RUN_TRAINING=True`로 설정하고 위에서부터 전체 실행하면 학습합니다. 기본값 `False`에서는 코드·구조·기록만 확인합니다.
3. 그래프와 Train / Validation 게이지는 실행 중 갱신됩니다. 검증은 공식 주기인 **1, 4, 8, …, 500 epoch**에 수행합니다. 검증하지 않은 epoch의 val 값은 빈칸입니다.
4. 중단 후 같은 RUN_NAME으로 다시 실행하면 마지막 저장된 optimizer update부터 이어집니다. 조건을 바꾸면 새 RUN_NAME을 사용합니다.

**이미 실행 중인 커널/프로세스에는 파일 수정이 자동 적용되지 않습니다.** 이전 실행의 결과는 새 조건의 결과로 합치지 않습니다.

기준 소스: [FraudGT 공식 저장소 고정 버전](https://github.com/junhongmit/FraudGT/tree/f88db9dc003a6f521a90631c4741541686043119). V10 입력을 사용한 비교 실험이며, 원 논문의 입력·전처리·여러 seed를 그대로 재현한 실험은 아닙니다.

In [ ]:
from pathlib import Path
import sys

RUN_NAME = "shared_corrected_001"
RUN_TRAINING = False
FOLLOW_ACTIVE_TRAINING = False  # 별도 프로세스로 이 RUN_NAME을 학습할 때만 True
RESOURCE_PROFILE = "memory_safe"  # "official": worker18/thread24; 동시 실행 시 메모리 확인 필요
GRAPH_MODE = "shared"
REFRESH_SECONDS = 2

# 두 파일은 동일한 버전의 실행 코드를 사용합니다.
PACKAGE_NAME = "V10_FraudGT_조건수정_20261027"
candidates = []
for parent in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([parent / PACKAGE_NAME, parent / "handoff" / PACKAGE_NAME])
PACKAGE_ROOT = next((p for p in candidates if (p / "corrected_runtime.py").is_file()), None)
if PACKAGE_ROOT is None:
    raise FileNotFoundError("handoff/V10_FraudGT_조건수정_20261027 폴더가 필요합니다.")
PACKAGE_ROOT = PACKAGE_ROOT.resolve()
for module_name in ("configure", "fraudgt_lab"):
    loaded = sys.modules.get(module_name)
    if loaded is not None and not Path(loaded.__file__).resolve().is_relative_to(PACKAGE_ROOT):
        raise RuntimeError("이전 실험 모듈이 남아 있습니다. 커널을 재시작하세요.")
sys.path.insert(0, str(PACKAGE_ROOT))
from configure import configure
EXPERIMENT_ROOT, RUN_ROOT = configure(RUN_NAME)
print("그래프 방식:", GRAPH_MODE)
print("실행 결과 경로:", RUN_ROOT)
print("실제 학습 실행:", RUN_TRAINING)

## 비교 조건과 수정한 차이

| 항목 | 개별 그래프 실습형 | 공유그래프 실습형 |
|---|---|---|
| 이웃 추출 | 실제 PyG LinkNeighborLoader, 무작위·비복원 [50, 50] | 동일 |
| 그래프 공유 | 거래마다 독립. 동일 계좌도 거래별 별도 복사 | 원래 2,048개 대상 거래의 배치 전체에서 계좌·간선 공유 |
| 메모리 처리 | 64개 독립 그래프씩 처리 후 2,048건 배치 손실 평균으로 합산 | 2,048건 공유 그래프 한 번에 처리 |
| 기본형 / PE 관계 | 정방향만 | 정방향만 |
| Multi 관계 | 정·역방향 + ego ID | 정·역방향 + ego ID |
| 학습 요청량 | 2,048 × 256 = 524,288건/epoch, 전체 학습 거래를 순회 | 동일 |
| 손실 / 누적 | BCE 가중치 [1, 6], 원래 배치 평균 8개 누적, clip=1 | 동일 |
| 검증 요청량 | 검증 전체 거래에서 균일 셔플, 최대 256 × 2,048건 | 동일 |
| 검증 / best 선택 | epoch 1,4,8,…500 / val F1@0.5 | 동일 |
| 그래프 기간 | train은 train까지, val은 train+val까지, test는 전체기간 | 동일 |

기존 코드의 기본형·PE 역방향 이웃 포함, 양성 전수+정상 표본 검증, 매 epoch 검증, 추가 query readout을 수정했습니다. 공유 범위 외의 주요 비교 조건은 일치시켰습니다. 개별 방식은 논문 기본 공유 방식에 대한 비교군입니다.

AdamW(lr=0.001, decay=1e-5), 5 epoch warmup+cosine, 총500 epoch, hidden64, 2 layers, 8 heads, attention dropout0.2를 사용합니다. 양성 가중치300은 사용하지 않습니다. 원 코드의 accumulation 경계 오류는 재현하지 않고 **완전한 8개 배치 평균을 누적**합니다. RNG와 초기화 호출 순서까지 원 저장소와 동일한 것은 아닙니다.

In [ ]:
import ast, fcntl, hashlib, itertools, json, math, time
from datetime import datetime, timedelta, timezone
from dataclasses import asdict, replace
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch_geometric.utils import softmax
from torch_geometric.sampler import NeighborSampler
from torch_geometric.loader import LinkNeighborLoader
from sklearn.metrics import average_precision_score
from IPython.display import display, HTML
from fraudgt_lab import PROJECT_ROOT, LEGACY_ROOT
from fraudgt_lab.config import TrainConfig, VARIANTS, EDGE_DIM, EDGE_INDICES, FORMAT_NAMES, digest, protocol
from fraudgt_lab.population import ASSETS, prepare_population, split_window
from fraudgt_lab.official_loader import build_index, MappedFeatureStore, MappedGraphStore, FORWARD, REVERSE
from fraudgt_lab.data import GraphBatch
from fraudgt_lab.training import save_checkpoint, finish_evaluation
from fraudgt_lab.storage import write_json, write_csv, log
from corrected_runtime import prepare, run_lock, require_verification, visible_functions, fingerprint
from graph_runtime import HIInputs
from common import G

workers, threads = (18, 24) if RESOURCE_PROFILE == "official" else (0, 2)
CONFIG = TrainConfig(graph_mode=GRAPH_MODE, loader_workers=workers, torch_threads=threads)
torch.set_num_threads(CONFIG.torch_threads)
require_verification()
display(pd.DataFrame([asdict(CONFIG)]).T.rename(columns={0: "설정값"}))
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "학습에는 CUDA GPU 필요")
print("공식 하드웨어 설정 대비:", "worker18/thread24" if RESOURCE_PROFILE == "official" else "worker0/thread2 메모리 절약 설정")

## 기간 분할과 피처

공식 방식의 일별 거래 수에 따른 약60/20/20 분할을 적용합니다. 라벨은 분할·그래프·피처에 넣지 않습니다.

| 구분 | 기간 | 대상 거래 수 |
|---|---|---:|
| Train | 2022-08-01 ~ 2022-09-25 | 102,505,233 |
| Validation | 2022-09-26 ~ 2022-10-13 | 34,491,749 |
| Test | 2022-10-14 ~ 2022-10-31 | 33,429,507 |

입력은 V10의 **계좌19종 + 거래수치14종 + 결제방식1종**입니다. 결제방식은 Unknown 포함8열 one-hot이므로 모델 거래 입력은22열입니다. PE의 port와 Multi의 ego는 구조 인코딩입니다. 기존 41종 결합모델에 있는 트리 전용 피처 전체를 GNN에 넣었다는 뜻은 아닙니다.

현재 장비의 동시 실행을 고려해 기본 worker0/thread2, CUDA allocator 상한은 프로세스당 GPU 메모리의35%입니다. 공식 worker18/thread24 프로필을 선택해도 메모리 사용은 별도로 확인해야 합니다. 샘플 그래프가 상한을 넘으면 자동으로 이웃 수나 공유 범위를 줄이지 않고 오류를 표시합니다.

관측값 중복 제거와 V10 스케일러를 유지합니다. 스케일러는 첫7일에 고정되었고, 이번 train에는 그 기간도 포함합니다. 일말 제한은 시연 조건이므로 학습 이웃에 적용하지 않습니다. 계좌 집계와 그래프는 **각 split 종료까지의 정보**를 사용해 해당 split 내부에서는 대상 거래 이후 정보도 포함하는 오프라인 조건입니다. train에는 val/test 간선을 넣지 않습니다. 정적 KYC의 실제 제공 시점은 별도 검증되지 않았습니다. 같은 분의 서로 다른 거래를 시간 cutoff로 강제 삭제하지 않으며, [50,50] 샘플링에서 선택되지 않는 경우는 존재합니다.

Test는 과거 프로젝트에서도 사용한 데이터이므로 새로운 외부 일반화 검증으로 해석하지 않습니다. 기존 HGB 결합 결과와 이번 GNN 비교를 섞지 않습니다.

In [ ]:
split_record = prepare_population()
display(pd.DataFrame(split_record["windows"]).T)
feature_spec = json.loads((EXPERIMENT_ROOT / "baseline_manifest.json").read_text())
display(pd.DataFrame({"node": pd.Series(feature_spec["node_names"]),
                      "edge": pd.Series(feature_spec["edge_names"] + ["payment_format"])}))

## 1. 기간 경계와 정답 조회

기간 경계는 거래 수만 사용합니다. 정답은 loss와 지표 계산 단계에서만 별도로 조회합니다. class9는 무패턴 평가용 구분이며 입력 피처가 아닙니다.

In [ ]:
def temporal_boundaries(counts):
    """Exact day-boundary objective used by upstream AMLDataset.process."""
    counts=np.asarray(counts,np.int64);total=int(counts.sum());target=np.array([.6,.2,.2])
    prefix=np.r_[0,np.cumsum(counts)]
    candidates=itertools.combinations(range(len(counts)),2)
    return min(candidates,key=lambda ij:max(abs(np.array([prefix[ij[0]],prefix[ij[1]]-prefix[ij[0]],total-prefix[ij[1]]])/total-target)/target))

class Labels:
    def __init__(self):
        self.ids=np.load(ASSETS/'positive_ids.npy');self.classes=np.load(ASSETS/'positive_classes.npy')
    def __call__(self,ids):
        ids=np.asarray(ids);p=np.searchsorted(self.ids,ids);valid=p<len(self.ids)
        found=np.flatnonzero(valid);found=found[self.ids[p[found]]==ids[found]]
        y=np.zeros(len(ids),np.int8);y[found]=self.classes[p[found]];return y

## 2. 전체 대상 거래 순회

두 방식은 동일한 seed와 거래 순서를 사용합니다. Train은 epoch마다 처음부터 반복하지 않고 전체 거래를 이어서 순회합니다. Val은 모든 검증 거래에서 라벨과 무관하게 최대524,288건을 균일 선택합니다.

In [ ]:
class PopulationStream:
    def __init__(self,config,scope='train'):
        self.config,self.scope=config,scope
        prepare_population()
        self.ids=np.load(ASSETS/f'{scope}_ids.npy',mmap_mode='r')
        self.batches_per_pass=math.ceil(len(self.ids)/config.seed_batch)
        self.current_cycle=None;self.order=None

    def permutation(self,cycle):
        if self.current_cycle==cycle:return self.order
        folder=ASSETS/'orders';folder.mkdir(exist_ok=True)
        path=folder/f'{self.scope}_seed{self.config.seed}_cycle{cycle}.npy'
        with path.with_suffix('.lock').open('a+') as lock:
            fcntl.flock(lock,fcntl.LOCK_EX)
            if not path.exists():
                values=np.array(self.ids,copy=True)
                np.random.default_rng(self.config.seed+cycle).shuffle(values)
                temp=path.with_suffix('.tmp.npy');np.save(temp,values);temp.replace(path)
        self.order=np.load(path,mmap_mode='r');self.current_cycle=cycle;return self.order

    def training_batches(self,epoch):
        result=[]
        for absolute in range((epoch-1)*self.config.iterations_per_epoch,epoch*self.config.iterations_per_epoch):
            cycle,index=divmod(absolute,self.batches_per_pass)
            result.append(np.asarray(self.permutation(cycle)[index*self.config.seed_batch:(index+1)*self.config.seed_batch],np.int64))
        return result

    def evaluation_ids(self,epoch=0,max_batches=None):
        """Shuffle the whole split, then take the loader budget; independent of labels."""
        limit=min(len(self.ids),(max_batches or self.config.validation_batches)*self.config.seed_batch)
        # Prefix of a uniform permutation; never oversample positive targets.
        seed=self.config.seed+epoch*1009+({'val':1,'test':2}[self.scope])*1000003
        positions=np.random.default_rng(seed).permutation(len(self.ids))[:limit]
        return np.asarray(self.ids[positions],np.int64)

## 3. 모델별 관계와 실제 PyG 이웃 추출

기본형/PE는 정방향만 전달합니다. Multi에만 역방향을 추가합니다. disjoint=True이면 거래별 독립 그래프, False이면 2,048건 배치 전체 공유 그래프입니다. CSC 캐시는 저장 방식일 뿐, 샘플링은 PyG가 수행합니다.

In [ ]:
def relations_for(variant):
    return [FORWARD, REVERSE] if variant == 'multi_fraudgt_format' else [FORWARD]

class PeriodGraph:
    def __init__(self, scope, variant='fraudgt_format', config=None):
        self.config = config or TrainConfig()
        self.scope, self.variant = scope, variant
        self.relations = relations_for(variant)
        self.disjoint = self.config.graph_mode == 'disjoint'
        folder = build_index(scope)
        self.metadata = json.loads((folder / 'complete.json').read_text())
        self.ids, self.arrays = {}, {}
        for kind, role in [(FORWARD, 'incoming'), (REVERSE, 'outgoing')]:
            if kind not in self.relations:
                continue
            # Copy-on-write maps are writable to Torch but never modify disk inputs.
            row = np.load(folder / f'{role}_row.npy', mmap_mode='c')
            ptr = np.load(folder / f'{role}_ptr.npy', mmap_mode='c')
            self.arrays[kind] = (torch.from_numpy(row), torch.from_numpy(ptr))
            self.ids[kind] = np.load(folder / f'{role}_ids.npy', mmap_mode='r')
        date = str(np.datetime64('2022-08-01') + np.timedelta64(split_window(scope)['last_day'], 'D'))
        self.nodes = np.load(G / 'nodes' / f'{date}.npy', mmap_mode='r')
        self.feature_store = MappedFeatureStore(self.nodes)
        self.graph_store = MappedGraphStore(self.arrays, self.metadata['nodes'])
        self.sampler = NeighborSampler((self.feature_store, self.graph_store),
                                      num_neighbors=[50, 50], replace=False,
                                      subgraph_type='directional', disjoint=self.disjoint)
        # Keep metadata iteration stable across process restarts, as HeteroData does.
        self.sampler.edge_types = list(self.relations)
        self.src, self.dst = [np.load(G / f'{name}.npy', mmap_mode='r') for name in ('src', 'dst')]

    def loader(self, ids, labels=None, batch_size=2048, shuffle=False, batch_sizes=None, seed=42):
        ids = np.asarray(ids, dtype=np.int64)
        edge_index = torch.from_numpy(np.stack([self.src[ids], self.dst[ids]]).astype(np.int64))
        labels = torch.zeros(len(ids)) if labels is None else torch.as_tensor(labels)
        options = {'batch_size': batch_size, 'shuffle': shuffle}
        if batch_sizes is not None:
            bounds = np.r_[0, np.cumsum(batch_sizes)]
            assert bounds[-1] == len(ids) and max(batch_sizes) <= batch_size
            options = {'batch_sampler': [range(int(a), int(b)) for a, b in zip(bounds[:-1], bounds[1:])]}
        return LinkNeighborLoader((self.feature_store, self.graph_store),
                                  num_neighbors=[50, 50], edge_label_index=(FORWARD, edge_index),
                                  edge_label=labels, **options,
                                  num_workers=self.config.loader_workers, neighbor_sampler=self.sampler,
                                  replace=False, disjoint=self.disjoint, subgraph_type='directional',
                                  filter_per_worker=False, generator=torch.Generator().manual_seed(seed))

## 4. V10 피처 연결

실제 샘플링된 거래 ID로 피처를 읽습니다. mmap 입출력을 위해 ID를 정렬한 뒤 원래 순서를 복원합니다. 개별 그래프에서 동일 거래 ID가 반복돼도 자기 대상 그래프의 간선만 판정에 연결합니다.

In [ ]:
class FeatureAdapter:
    def __init__(self, use_ports=False):
        self.inputs = HIInputs()
        self.formats = np.load(G / 'fmt.npy', mmap_mode='r')
        port_root = PROJECT_ROOT / 'experiments/v10_fraudgt_compare_20261026/cache'
        self.port_arrays = ([np.load(port_root / f'{role}_ports.npy', mmap_mode='r')
                             for role in ('incoming', 'outgoing')] if use_ports else [])

    def edge_features(self, ids):
        # Read mmap rows in increasing order, once per transaction. Restore the
        # exact sampled order afterwards; no graph edge or feature is changed.
        ordered, restore = np.unique(ids, return_inverse=True)
        numeric = self.inputs.edges(ordered)[:, EDGE_INDICES]
        code = np.asarray(self.formats[ordered], dtype=np.int64)
        code = np.where((code >= 0) & (code < 7), code, 7)
        values = np.column_stack([numeric, np.eye(len(FORMAT_NAMES), dtype=np.float32)[code]])
        return np.ascontiguousarray(values[restore], dtype=np.float32)

    def ports(self, ids):
        if not self.port_arrays:
            return np.zeros((len(ids), 2), dtype=np.float32)
        ordered, restore = np.unique(ids, return_inverse=True)
        return np.column_stack([a[ordered] for a in self.port_arrays])[restore].astype(np.float32)

    def materialize(self, graph, sampled, seed_ids):
        rows = sampled[FORWARD].input_id.numpy()
        targets = np.asarray(seed_ids[rows], dtype=np.int64)
        forward_ids = np.asarray(graph.ids[FORWARD][sampled[FORWARD].e_id.numpy()], dtype=np.int64)
        has_reverse = REVERSE in graph.relations
        reverse_ids = (np.asarray(graph.ids[REVERSE][sampled[REVERSE].e_id.numpy()], dtype=np.int64)
                       if has_reverse else np.empty(0, dtype=np.int64))
        all_ids = np.concatenate([forward_ids, reverse_ids])
        nodes = sampled['node'].n_id.numpy()
        indices = np.concatenate([sampled[FORWARD].edge_index.numpy(),
                                  sampled[REVERSE].edge_index.numpy()[::-1] if has_reverse else np.empty((2,0),dtype=np.int64)], axis=1)
        target_index = sampled[FORWARD].edge_label_index.numpy()
        assert np.array_equal(nodes[target_index[0]], graph.src[targets])
        assert np.array_equal(nodes[target_index[1]], graph.dst[targets])
        assert np.array_equal(nodes[indices[0]], graph.src[all_ids])
        assert np.array_equal(nodes[indices[1]], graph.dst[all_ids])
        if graph.disjoint:
            groups = sampled['node'].batch.numpy()
            assert np.array_equal(groups[target_index[0]], groups[target_index[1]])
            assert np.array_equal(groups[indices[0]], groups[indices[1]]), 'Edges must not cross target graphs'
        else:
            groups = np.zeros(len(nodes), dtype=np.int64)
            assert len(np.unique(nodes)) == len(nodes), 'Shared batch must deduplicate nodes'
            assert len(np.unique(forward_ids)) == len(forward_ids)
        ego = np.zeros(len(nodes), dtype=np.float32)
        ego[np.unique(target_index)] = 1
        node_features = np.column_stack([sampled['node'].x.numpy(), ego]).astype(np.float32)
        positions = np.full(len(targets), -1, dtype=np.int64)
        width = len(graph.src)
        context_keys = groups[indices[1,:len(forward_ids)]] * width + forward_ids
        target_keys = groups[target_index[0]] * width + targets
        order = np.argsort(context_keys)
        offset = np.searchsorted(context_keys[order], target_keys)
        valid = offset < len(order)
        found = np.flatnonzero(valid)
        found = found[context_keys[order[offset[found]]] == target_keys[found]]
        positions[found] = order[offset[found]]
        arrays = [node_features, indices, self.edge_features(all_ids), target_index,
                  self.edge_features(targets), np.arange(len(all_ids)) < len(forward_ids),
                  np.arange(len(all_ids)) >= len(forward_ids), positions,
                  self.ports(all_ids), self.ports(targets)]
        batch = GraphBatch(*[torch.from_numpy(np.ascontiguousarray(v)) for v in arrays])
        stats = {'targets': len(targets), 'nodes': len(nodes), 'forward_edges': len(forward_ids),
                 'reverse_edges': len(reverse_ids), 'target_edges_not_sampled': int((positions < 0).sum()),
                 'unique_nodes': len(np.unique(nodes)), 'disjoint': graph.disjoint,
                 'relations': len(graph.relations)}
        return rows, batch, stats

## 5. FraudGT attention block

계좌 query/key/value와 거래 속성의 bias·gate를 이용해 이웃 메시지를 합산합니다. hidden64 / heads8이며 정방향·역방향 관계별 파라미터를 구분합니다. forward_queries는 선택적 진단용이고 기본 공식 readout에서는 사용하지 않습니다.

In [ ]:
class FraudGTBlock(nn.Module):
    """Sparse attention, attribute bias, gated values and updated edge embeddings."""

    def __init__(self, relations: int, hidden: int = 64, heads: int = 8, attention_dropout: float = 0.2):
        super().__init__()
        self.hidden = hidden
        self.heads = heads
        self.node_norm = nn.LayerNorm(hidden)
        self.node_ffn_norm = nn.LayerNorm(hidden)
        self.edge_norm = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(relations)])
        self.query = nn.Linear(hidden, hidden)
        self.key = nn.Linear(hidden, hidden)
        self.value = nn.Linear(hidden, hidden)
        self.node_output = nn.Linear(hidden, hidden)
        self.edge_bias = nn.ModuleList([nn.Linear(hidden, hidden) for _ in range(relations)])
        self.edge_gate = nn.ModuleList([nn.Linear(hidden, hidden) for _ in range(relations)])
        self.edge_output = nn.ModuleList([nn.Linear(hidden, hidden) for _ in range(relations)])
        self.relation_keys = nn.Parameter(torch.empty(relations, heads, hidden // heads, hidden // heads))
        nn.init.xavier_uniform_(self.relation_keys)
        self.node_ffn = nn.Sequential(nn.Linear(hidden, hidden * 2), nn.GELU(), nn.Linear(hidden * 2, hidden))
        self.edge_ffn = nn.ModuleList([nn.Sequential(nn.Linear(hidden, hidden * 2), nn.GELU(), nn.Linear(hidden * 2, hidden)) for _ in range(relations)])
        self.attention_dropout = nn.Dropout(attention_dropout)

    def forward(self, nodes: torch.Tensor, indices: list, edges: list):
        normalized = self.node_norm(nodes)
        shape = (-1, self.heads, self.hidden // self.heads)
        query = self.query(normalized).view(shape)
        key = self.key(normalized).view(shape)
        value = self.value(normalized).view(shape)
        destinations, scores, messages, next_edges = [], [], [], []
        for relation, (index, attributes) in enumerate(zip(indices, edges)):
            source, destination = index
            edge_normalized = self.edge_norm[relation](attributes)
            # Transform each node once per relation, instead of allocating an E×H×D×D tensor.
            transformed_key = torch.einsum("hij,nhj->nhi", self.relation_keys[relation], key)
            vector_score = query[destination] * transformed_key[source] + self.edge_bias[relation](edge_normalized).view(shape)
            scalar_score = (vector_score.sum(-1) / math.sqrt(self.hidden // self.heads)).clamp(-5, 5)
            message = value[source] * self.edge_gate[relation](edge_normalized).view(shape).sigmoid()
            updated = attributes + self.edge_output[relation](vector_score.reshape(-1, self.hidden))
            updated = updated + self.edge_ffn[relation](updated)
            destinations.append(destination)
            scores.append(scalar_score)
            messages.append(message)
            next_edges.append(updated)
        destination = torch.cat(destinations)
        attention = softmax(torch.cat(scores), destination, num_nodes=len(nodes))
        message = self.attention_dropout(attention).unsqueeze(-1) * torch.cat(messages)
        aggregate = nodes.new_zeros((len(nodes), self.heads, self.hidden // self.heads))
        aggregate.index_add_(0, destination, message)
        updated_nodes = nodes + self.node_output(aggregate.flatten(1))
        # Preserve the published code's normalization/FFN ordering exactly.
        normalized_nodes = self.node_ffn_norm(updated_nodes)
        updated_nodes = normalized_nodes + self.node_ffn(normalized_nodes)
        return updated_nodes, next_edges

    def forward_queries(self, nodes, indices, edges, target_index, target_edges):
        """Read every seed without adding unsampled edges to message passing.

        Use the same relation-0 edge update as the published block. This repairs
        the upstream readout's omission of seeds absent from sampled e_id.
        No new parameters, neighbors, attention edges or labels are introduced.
        """
        updated_nodes, next_edges = self.forward(nodes, indices, edges)
        normalized = self.node_norm(nodes)
        shape = (-1, self.heads, self.hidden // self.heads)
        query = self.query(normalized).view(shape)
        key = self.key(normalized).view(shape)
        transformed = torch.einsum('hij,nhj->nhi', self.relation_keys[0], key)
        source, destination = target_index
        edge_normalized = self.edge_norm[0](target_edges)
        vector = query[destination] * transformed[source] + self.edge_bias[0](edge_normalized).view(shape)
        target_edges = target_edges + self.edge_output[0](vector.reshape(-1, self.hidden))
        target_edges = target_edges + self.edge_ffn[0](target_edges)
        return updated_nodes, next_edges, target_edges

## 6. 기본형 · PE · Multi 모델

공식 readout은 샘플 그래프에 실제 포함된 대상 간선만 판정합니다. 반환값 selection으로 요청 거래와 지표를 정확히 연결합니다. 모든 요청 거래를 별도 query로 판정하는 all_targets는 공식 비교 기본값이 아닙니다.

In [ ]:
class FraudGT(nn.Module):
    """Basic, PE and Multi variants; 34 business features in every variant."""

    def __init__(self, variant: str, readout="official"):
        super().__init__()
        self.variant = variant
        self.readout = readout
        self.use_ports = variant in ("pe_fraudgt_format", "multi_fraudgt_format")
        self.reverse = variant == "multi_fraudgt_format"
        # Official defaults set add_ego_id=False; only the Multi YAML enables it.
        self.use_ego = self.reverse
        relations = 2 if self.reverse else 1
        port_columns = 1 if self.reverse else (2 if self.use_ports else 0)
        self.node_encoder = nn.Linear(20 if self.use_ego else 19, 64)
        self.edge_encoder = nn.ModuleList([nn.Linear(EDGE_DIM + port_columns, 64) for _ in range(relations)])
        self.layers = nn.ModuleList([FraudGTBlock(relations) for _ in range(2)])
        self.head = nn.Sequential(nn.Linear(192, 192), nn.ReLU(), nn.Linear(192, 1))
        self.checkpoint_layers = True

    def forward(self, batch: GraphBatch):
        masks = [batch.incoming] + ([batch.outgoing] if self.reverse else [])
        indices, edges = [], []
        for relation, mask in enumerate(masks):
            index = batch.edge_index[:, mask]
            if relation == 1:
                index = index.flip(0)
            attributes = batch.edge_features[mask]
            if self.use_ports:
                ports = batch.ports[mask, relation:relation + 1] if self.reverse else batch.ports[mask]
                attributes = torch.cat([attributes, ports], dim=1)
            indices.append(index)
            edges.append(self.edge_encoder[relation](attributes))
        nodes = self.node_encoder(batch.node_features if self.use_ego else batch.node_features[:, :19])
        if self.readout == 'official':
            # Same sampled-edge mask/order as upstream hetero_edge._apply_index.
            present = torch.nonzero(batch.target_edge_positions >= 0).flatten()
            selection = present[torch.argsort(batch.target_edge_positions[present])]
            for layer in self.layers:
                if self.training and self.checkpoint_layers:
                    from torch.utils.checkpoint import checkpoint
                    nodes, edges = checkpoint(layer, nodes, indices, edges, use_reentrant=False, preserve_rng_state=True)
                else:
                    nodes, edges = layer(nodes, indices, edges)
            positions = batch.target_edge_positions[selection]
            source, destination = batch.target_index[:, selection]
            representation = torch.cat([nodes[source], nodes[destination], edges[0][positions]], dim=1)
        elif self.readout == 'all_targets':
            # Optional deployment/diagnostic extension, disabled for the official comparison.
            attributes = batch.target_features
            if self.use_ports:
                ports = batch.target_ports[:, :1] if self.reverse else batch.target_ports
                attributes = torch.cat([attributes, ports], dim=1)
            queries = self.edge_encoder[0](attributes)
            for layer in self.layers:
                if self.training and self.checkpoint_layers:
                    from torch.utils.checkpoint import checkpoint
                    nodes, edges, queries = checkpoint(layer.forward_queries, nodes, indices, edges,
                        batch.target_index, queries, use_reentrant=False, preserve_rng_state=True)
                else:
                    nodes, edges, queries = layer.forward_queries(nodes, indices, edges, batch.target_index, queries)
            source, destination = batch.target_index
            representation = torch.cat([nodes[source], nodes[destination], queries], dim=1)
            selection = torch.arange(len(source), device=nodes.device)
        else:
            raise ValueError(self.readout)
        return self.head(representation).squeeze(-1), selection

def make_model(variant, config=None):
    if variant not in ('fraudgt_format', 'pe_fraudgt_format', 'multi_fraudgt_format'):
        raise ValueError(variant)
    model = FraudGT(variant, readout=config.readout if config else 'official')
    if config is not None:
        model.checkpoint_layers = config.checkpoint_layers
    return model

In [ ]:
for variant in VARIANTS:
    model = make_model(variant, CONFIG)
    print("\n" + variant, "parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
    print(model)
    del model

## 7. 손실과 optimizer

양성6 / 정상1 가중치 BCE입니다. 정상 표본 IPW는 사용하지 않습니다. AdamW decay 제외 규칙도 원 코드와 맞춥니다.

In [ ]:
def class_weighted_loss(logits, labels, config):
    """Upstream binary branch, before mean reduction; no sampling weight here."""
    weights = torch.where(labels > 0, config.positive_weight, 1.0)
    return F.binary_cross_entropy_with_logits(logits, labels, weight=weights, reduction='none')

def make_optimizer(model, config):
    named = list(model.named_parameters())
    no_decay = ('bias', 'LayerNorm.bias', 'LayerNorm.weight')
    groups = [
        {'params': [p for name, p in named if not any(key in name for key in no_decay)],
         'weight_decay': config.weight_decay},
        {'params': [p for name, p in named if any(key in name for key in no_decay)],
         'weight_decay': 0.0},
    ]
    return torch.optim.AdamW(groups, lr=config.learning_rate(1), weight_decay=config.weight_decay)

## 8. 정확도·탐지율과 판정 범위

ACC / P / R / F1 / AP / 무패턴 Recall은 실제 판정된 거래 기준입니다. coverage는 판정수÷요청수이며, missing_positive는 요청했지만 판정되지 않은 양성 수입니다. requested_recall_lower_bound는 이 거래들을 미탐으로 계산한 요청 집합의 Recall 하한입니다. 이 값도 전체 데이터 전수 성능은 아닙니다.

In [ ]:
class EpochMetrics:
    def __init__(self):
        self.requested=0;self.requested_positive=0;self.requested_unmatched=0
        self.loss_sum=0.;self.labels=[];self.logits=[]
    def add_requested(self,labels):
        self.requested+=len(labels);self.requested_positive+=int((labels>0).sum());self.requested_unmatched+=int((labels==9).sum())
    def add_scored(self,labels,logits,loss_sum):
        self.labels.append(np.asarray(labels,np.int8));self.logits.append(np.asarray(logits,np.float32));self.loss_sum+=float(loss_sum)
    def state(self):
        return {'requested':self.requested,'requested_positive':self.requested_positive,'requested_unmatched':self.requested_unmatched,
                'loss_sum':self.loss_sum,'labels':self.labels,'logits':self.logits}
    @classmethod
    def restore(cls,state):
        obj=cls()
        if state is not None:obj.__dict__.update(state)
        return obj
    def compute(self):
        y=np.concatenate(self.labels) if self.labels else np.empty(0,np.int8)
        scores=np.concatenate(self.logits) if self.logits else np.empty(0,np.float32)
        positive=y>0;pred=scores>0
        tp=int((positive&pred).sum());fp=int((~positive&pred).sum());fn=int((positive&~pred).sum())
        ratio=lambda a,b:float(a/b) if b else 0.
        unmatched=y==9;known=positive&~unmatched
        return {'loss':ratio(self.loss_sum,len(y)),'accuracy_at_05':float((positive==pred).mean()) if len(y) else 0.,
                'precision_at_05':ratio(tp,tp+fp),'recall_at_05':ratio(tp,tp+fn),'F1':ratio(2*tp,2*tp+fp+fn),
                'AP':float(average_precision_score(positive,scores)) if positive.any() else 0.,
                'unmatched_recall_at_05':ratio(int((unmatched&pred).sum()),int(unmatched.sum())),
                'known_recall_at_05':ratio(int((known&pred).sum()),int(known.sum())),
                'requested':self.requested,'scored':len(y),'coverage':ratio(len(y),self.requested),
                'requested_positive':self.requested_positive,'scored_positive':int(positive.sum()),
                'requested_recall_lower_bound':ratio(tp,self.requested_positive),
                'requested_unmatched_recall_lower_bound':ratio(int((unmatched&pred).sum()),self.requested_unmatched),
                'missing_positive':self.requested_positive-int(positive.sum()),'missing_targets':self.requested-len(y)}

## 9. 실시간 진행 기록

Train과 Validation에서 처리한 요청 거래 수를 실시간으로 저장합니다. 게이지100%는 요청한 배치를 모두 처리했다는 뜻이며, 모든 거래에 점수를 냈다는 뜻은 아닙니다. 실제 판정 건수도 나란히 표시합니다.

In [ ]:
class EpochProgressReporter:
    def __init__(self,folder,variant,epoch,config,train_total,val_total,callback=None):
        self.path=folder/'live_progress.json';self.callback=callback;self.last=0.;self.started={}
        self.value={'variant':variant,'epoch':epoch,'epochs':config.epochs,'phase':'train',
                    'train':{'completed':0,'total':train_total,'percent':0.,'loss':None},
                    'validation':{'completed':0,'total':val_total,'percent':0.,'scheduled':config.validation_epoch(epoch)}}
    def __call__(self,phase,completed,total,loss=None,scored=None,force=False):
        now=time.monotonic();self.started.setdefault(phase,now)
        self.value['phase']=phase
        self.value[phase].update(completed=int(completed),total=int(total),percent=100*completed/total if total else 0.,
                                 elapsed_seconds=now-self.started[phase])
        if loss is not None:self.value[phase]['loss']=float(loss)
        if scored is not None:self.value[phase]['scored']=int(scored)
        if force or now-self.last>=1:
            self.value['utc']=datetime.now(timezone.utc).isoformat();write_json(self.path,self.value);self.last=now
            if self.callback:self.callback(self.value)

## 10. 재개 가능한 샘플링과 손실 누적

원래2,048건 배치를 기본 단위로 유지합니다. 개별 그래프의64건 조각은 loss 합을 역전파한 뒤 실제 판정 건수로 나눠 한 배치 평균을 만들고, 그 평균8개를 누적합니다. 공유그래프는 조각으로 나누지 않습니다.

In [ ]:
def sample_next(iterator,config,scope,epoch,index):
    key=f'{config.seed}|{scope}|{epoch}|{index}'.encode()
    seed=int.from_bytes(hashlib.blake2b(key,digest_size=8).digest(),'little')%(2**63-1)
    with torch.random.fork_rng(devices=[]):
        torch.random.default_generator.manual_seed(seed)
        return next(iterator)

def graph_chunks(logical_sizes,config):
    """Reference keeps each target disjoint; shared mode keeps the whole seed batch."""
    groups=[];sizes=[]
    for size in logical_sizes:
        parts=[min(config.graph_batch_size,size-first) for first in range(0,size,config.graph_batch_size)]
        groups.append(len(parts));sizes.extend(parts)
    return groups,sizes

def combine_batch_gradients(model,previous,scored):
    """Add one complete loader-batch MEAN to accumulated previous batch means."""
    for parameter,old in zip(model.parameters(),previous):
        current=parameter.grad
        if current is not None:
            if scored<=0:raise RuntimeError('Gradient without any scored target')
            current.div_(scored)
        if old is not None:
            if current is None:parameter.grad=old
            else:current.add_(old)

## 11. 한 epoch 학습

매 optimizer update 경계에서 체크포인트와 진행 기록을 남깁니다. 마지막 저장 이후 진행만 다시 계산하므로 중단 시 전체 epoch를 버리지 않습니다.

In [ ]:
def train_one_epoch(model,stream,graph,adapter,labels,optimizer,device,epoch,config,
                    start_batch=0,metric_state=None,save_progress=None,reporter=None):
    model.train();batches=stream.training_batches(epoch);total=sum(map(len,batches))
    metrics=EpochMetrics.restore(metric_state)
    if reporter:reporter('train',metrics.requested,total,scored=sum(map(len,metrics.labels)),force=True)
    if start_batch==len(batches):return metrics
    ids=np.concatenate(batches[start_batch:]);y=labels(ids)
    groups,sizes=graph_chunks([len(b) for b in batches[start_batch:]],config)
    loader=graph.loader(ids,(y>0).astype(np.float32),config.graph_batch_size,batch_sizes=sizes,seed=config.seed+epoch)
    iterator=iter(loader);optimizer.zero_grad(set_to_none=True)
    graph_index=sum(graph_chunks([len(b) for b in batches[:start_batch]],config)[0])
    for logical in range(start_batch,config.iterations_per_epoch):
        previous=[parameter.grad for parameter in model.parameters()]
        optimizer.zero_grad(set_to_none=True);batch_scored=0
        for _ in range(groups[logical-start_batch]):
            sampled=sample_next(iterator,config,'train',epoch,graph_index);graph_index+=1
            rows,batch,_=adapter.materialize(graph,sampled,ids)
            prediction,selection=model(batch.to(device));selected=selection.detach().cpu().numpy()
            metrics.add_requested(y[rows]);truth=y[rows[selected]]
            if len(prediction):
                target=torch.as_tensor(truth>0,dtype=torch.float32,device=device)
                losses=class_weighted_loss(prediction,target,config)
                if not torch.isfinite(losses).all():raise FloatingPointError('Nonfinite training loss')
                losses.sum().backward()
                metrics.add_scored(truth,prediction.detach().cpu().numpy(),float(losses.detach().sum()))
                batch_scored+=len(prediction)
            if reporter:
                scored=sum(map(len,metrics.labels))
                reporter('train',metrics.requested,total,metrics.loss_sum/scored if scored else None,scored)
            del batch,sampled,prediction,selection
        combine_batch_gradients(model,previous,batch_scored)
        cursor=logical+1
        if cursor%config.batch_accumulation==0 or cursor==config.iterations_per_epoch:
            norm=torch.nn.utils.clip_grad_norm_(model.parameters(),config.gradient_clip)
            if not torch.isfinite(norm):raise FloatingPointError('Nonfinite gradient')
            optimizer.step();optimizer.zero_grad(set_to_none=True)
            if save_progress:save_progress(cursor,metrics)
    assert metrics.requested==total
    if reporter:
        result=metrics.compute();reporter('train',total,total,result['loss'],result['scored'],True)
    return metrics

## 12. 검증

검증에는 optimizer step이 없습니다. train+val 기간 그래프를 사용하고 지정된 검증 거래를 중복 없이 처리합니다. 실제 판정 범위와 누락도 기록합니다.

In [ ]:
def evaluate_one_epoch(model,ids,graph,adapter,labels,device,config,epoch=0,reporter=None):
    model.eval();y=labels(ids);metrics=EpochMetrics();visited=np.zeros(len(ids),bool)
    loader=graph.loader(ids,(y>0).astype(np.float32),batch_size=config.graph_batch_size,shuffle=False)
    iterator=iter(loader)
    if reporter:reporter('validation',0,len(ids),scored=0,force=True)
    with torch.inference_mode():
        for index in range(len(loader)):
            sampled=sample_next(iterator,config,graph.scope,epoch,index)
            rows,batch,_=adapter.materialize(graph,sampled,ids)
            assert not visited[rows].any();visited[rows]=True
            logits,selection=model(batch.to(device));selected=selection.cpu().numpy()
            metrics.add_requested(y[rows]);truth=y[rows[selected]]
            if len(logits):
                if not torch.isfinite(logits).all():raise FloatingPointError('Nonfinite validation score')
                target=torch.as_tensor(truth>0,dtype=torch.float32,device=device)
                losses=class_weighted_loss(logits,target,config)
                metrics.add_scored(truth,logits.cpu().numpy(),float(losses.sum()))
            if reporter:reporter('validation',metrics.requested,len(ids),scored=sum(map(len,metrics.labels)),force=metrics.requested==len(ids))
            del batch,sampled,logits,selection
    assert visited.all() and metrics.requested==len(ids)
    return metrics.compute()

## 13. epoch 표와 예상시간

검증하지 않은 epoch에 이전 val 값을 복사하지 않습니다. ETA는 train과 val 시간을 따로 평균내고 남은 공식 검증 횟수를 반영합니다. 표시 시간은 현재 모델 하나의 예상시간입니다.

In [ ]:
def history_row(epoch,train,val,config,train_seconds,val_seconds,partial):
    row={'epoch':epoch,'learning_rate':config.learning_rate(epoch),'selection_eligible':config.selection_epoch(epoch),
         'validation_performed':val is not None,'train_seconds':train_seconds,'validation_seconds':val_seconds,
         'seconds_this_process_segment':train_seconds+val_seconds,'resumed_partial_epoch':partial,
         'train_loss':train['loss'],'training_rows':train['requested'],'training_scored':train['scored'],
         'validation_rows':val['requested'] if val else 0,'validation_scored':val['scored'] if val else 0,
         'optimizer_updates':(config.iterations_per_epoch+config.batch_accumulation-1)//config.batch_accumulation}
    for prefix,values in [('train',train),('val',val)]:
        for key in ['loss','accuracy_at_05','precision_at_05','recall_at_05','F1','AP','unmatched_recall_at_05',
                    'known_recall_at_05','coverage','requested_recall_lower_bound','missing_positive','missing_targets']:
            row[prefix+'_'+key]=values[key] if values else None
    return row

def estimate_eta(history,config):
    completed=max((row['epoch'] for row in history),default=0)
    train_times=[row['train_seconds'] for row in history if not row['resumed_partial_epoch']][-5:]
    val_times=[row['validation_seconds'] for row in history if row['validation_performed']][-5:]
    if not train_times:return {'mean_epoch_seconds':None,'remaining_hours':None,'estimated_finish_utc':None}
    train_mean=float(np.mean(train_times));val_mean=float(np.mean(val_times)) if val_times else 0.
    remaining_epochs=config.epochs-completed
    remaining_evaluations=sum(config.validation_epoch(e) for e in range(completed+1,config.epochs+1))
    seconds=train_mean*remaining_epochs+val_mean*remaining_evaluations
    return {'mean_epoch_seconds':train_mean+val_mean/config.eval_period,'train_mean_seconds':train_mean,
            'validation_mean_seconds':val_mean,'remaining_validation_epochs':remaining_evaluations,
            'remaining_hours':seconds/3600,'estimated_finish_utc':(datetime.now(timezone.utc)+timedelta(seconds=seconds)).isoformat(),
            'scope':'current variant only; accounts for scheduled validation'}

## 14. 모델별 전체 학습과 재개

세 모델을 순서대로 실행합니다. best는 지정 검증 epoch의 F1@0.5로만 선택합니다. 마지막 체크포인트의 optimizer·RNG·진행 위치·누적 지표를 함께 복원합니다.

In [ ]:
def run_variant(config,variant,on_epoch=None,on_progress=None):
    folder=RUN_ROOT/'models'/variant;folder.mkdir(parents=True,exist_ok=True)
    if (folder/'complete.json').exists():return json.loads((folder/'complete.json').read_text())
    torch.set_num_threads(config.torch_threads);torch.cuda.set_per_process_memory_fraction(.35);torch.manual_seed(config.seed)
    device=torch.device('cuda');model=make_model(variant,config).to(device);optimizer=make_optimizer(model,config)
    protocol_hash=digest(RUN_ROOT/'protocol.json')
    state={'epoch':1,'loader_cursor':0,'history':[],'best':{'F1':-1.,'epoch':0},'metric_state':None}
    if (folder/'last.pt').exists():
        state=torch.load(folder/'last.pt',map_location='cpu',weights_only=False)
        assert state['protocol_sha256']==protocol_hash,'Changed protocol cannot resume old checkpoints'
        model.load_state_dict(state['model']);optimizer.load_state_dict(state['optimizer'])
        torch.set_rng_state(state['rng_cpu']);torch.cuda.set_rng_state_all(state['rng_cuda'])
    history,best=state['history'],state['best'];labels=Labels()
    train_stream,val_stream=PopulationStream(config,'train'),PopulationStream(config,'val')
    train_graph,val_graph=PeriodGraph('train',variant,config),PeriodGraph('val',variant,config)
    adapter=FeatureAdapter(variant!='fraudgt_format');(folder/'model_structure.txt').write_text(str(model))
    def checkpoint(epoch,cursor,metric_state):
        save_checkpoint(folder/'last.pt',{'model':model.state_dict(),'optimizer':optimizer.state_dict(),
            'epoch':epoch,'loader_cursor':cursor,'metric_state':metric_state,'history':history,'best':best,
            'protocol_sha256':protocol_hash,'rng_cpu':torch.get_rng_state(),'rng_cuda':torch.cuda.get_rng_state_all()})
    log('corrected_training_started',variant=variant,graph_mode=config.graph_mode,start_epoch=state['epoch'])
    try:
        for epoch in range(state['epoch'],config.epochs+1):
            for group in optimizer.param_groups:group['lr']=config.learning_rate(epoch)
            cursor=state['loader_cursor'] if epoch==state['epoch'] else 0
            metric_state=state['metric_state'] if epoch==state['epoch'] else None
            total=sum(map(len,train_stream.training_batches(epoch)))
            val_total=min(len(val_stream.ids),config.validation_batches*config.seed_batch)
            reporter=EpochProgressReporter(folder,variant,epoch,config,total,val_total,on_progress)
            started=time.monotonic()
            def save_progress(next_batch,metrics):
                checkpoint(epoch,next_batch,metrics.state())
                result=metrics.compute();value={'epoch':epoch,'loader_batches':next_batch,'epoch_targets':metrics.requested,
                       'train_loss':result['loss'],'scored_targets':result['scored'],'coverage':result['coverage']}
                write_json(folder/'progress.json',value);log('corrected_training_progress',variant=variant,**value)
            train=train_one_epoch(model,train_stream,train_graph,adapter,labels,optimizer,device,epoch,config,
                                  cursor,metric_state,save_progress,reporter).compute()
            train_seconds=time.monotonic()-started;val=None;val_seconds=0.
            if config.validation_epoch(epoch):
                log('corrected_validation_started',variant=variant,epoch=epoch)
                val_start=time.monotonic();ids=val_stream.evaluation_ids(epoch)
                val=evaluate_one_epoch(model,ids,val_graph,adapter,labels,device,config,epoch,reporter)
                val_seconds=time.monotonic()-val_start
                if config.selection_epoch(epoch) and val['F1']>best['F1']:
                    best={'F1':val['F1'],'epoch':epoch}
                    save_checkpoint(folder/'best.pt',{'state_dict':model.state_dict(),'variant':variant,'epoch':epoch,
                                    'config':asdict(config),'protocol_sha256':protocol_hash,'validation':val})
            row=history_row(epoch,train,val,config,train_seconds,val_seconds,bool(cursor));history.append(row)
            write_json(folder/'history.json',history);write_csv(folder/'epoch_metrics.csv',history)
            checkpoint(epoch+1,0,None)
            if epoch%config.checkpoint_every(variant)==0 or epoch==config.epochs:
                save_checkpoint(folder/f'epoch{epoch}.pt',{'state_dict':model.state_dict(),'config':asdict(config),
                                'epoch':epoch,'protocol_sha256':protocol_hash})
            eta=estimate_eta(history,config);write_json(folder/'eta.json',eta)
            log('corrected_epoch_complete',variant=variant,best_epoch=best['epoch'],**row)
            if on_epoch:on_epoch(history,variant,best['epoch'],eta)
        result={'status':'complete','variant':variant,'epochs':len(history),'best':best}
        write_json(folder/'complete.json',result);return result
    finally:
        del model,optimizer,train_graph,val_graph,adapter;torch.cuda.empty_cache()

## 15. 게이지와 그래프 표시

화면 출력은 같은 영역을 갱신합니다. Train loss는 진행 중 갱신하고, val 지표 곡선은 해당 검증이 끝났을 때 추가합니다. 검증을 건너뛴 epoch에는 빈칸과 안내 문구가 보입니다. PNG를 별도로 저장하지 않습니다.

In [ ]:
"""Notebook display helpers; no model training or file writes."""
import json,time,fcntl
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display,HTML

METRIC_COLUMNS=['epoch','train_loss','train_accuracy_at_05','train_precision_at_05',
                'train_recall_at_05','train_F1','train_unmatched_recall_at_05','train_AP',
                'val_loss','val_accuracy_at_05',
                'val_precision_at_05','val_recall_at_05','val_F1','val_unmatched_recall_at_05','val_AP',
                'train_coverage','val_coverage','val_missing_positive',
                'val_requested_recall_lower_bound','validation_performed','learning_rate']

def read_result_json(path,default=None):
    try:return json.loads(Path(path).read_text())
    except (FileNotFoundError,json.JSONDecodeError):return default


def training_is_active(run_dir):
    path=Path(run_dir)/'run.lock'
    if not path.exists():return False
    with path.open('rb') as handle:
        try:fcntl.flock(handle,fcntl.LOCK_EX|fcntl.LOCK_NB)
        except BlockingIOError:return True
        fcntl.flock(handle,fcntl.LOCK_UN);return False


def progress_bar_html(live,config,variant,active):
    from html import escape
    epoch=live.get('epoch',1);phase=live.get('phase','대기')
    text='실행 중' if active else '저장된 기록 · 현재 실행 중 아님'
    out=f'<div style="padding:18px;border:1px solid #ddd;border-radius:10px;font-family:system-ui"><b>{escape(variant)} · Epoch {epoch}/{config.epochs}</b><p>{text}</p>'
    for key,label,color in [('train','Train','#3478f6'),('validation','Validation','#ef922d')]:
        value=live.get(key,{})
        if key=='validation' and not value.get('scheduled',config.validation_epoch(epoch)):
            out+='<p><b>Validation: 이번 epoch는 공식 검증 주기가 아닙니다.</b></p>';continue
        done=int(value.get('completed',0));total=int(value.get('total',config.seed_batch*(config.iterations_per_epoch if key=='train' else config.validation_batches)))
        total=max(1,total);percent=100*done/total
        scored=value.get('scored');scored_text=f' · 실제 판정 {scored:,}건' if scored is not None else ''
        out+=f'<p><b>{label}</b> {done:,}/{total:,}건 · {percent:.2f}%{scored_text}</p><progress value="{done}" max="{total}" style="width:100%;height:22px;accent-color:{color}"></progress>'
    return out+'</div>'


def learning_figure(history,variant,live=None):
    fig,axes=plt.subplots(2,2,figsize=(14,8));frame=pd.DataFrame(history)
    fig.suptitle(variant+' | official scored-target metrics')
    if len(frame):
        axes[0,0].plot(frame.epoch,frame.train_loss,label='Train loss')
        valid=frame[frame.validation_performed]
        if len(valid):
            axes[0,0].plot(valid.epoch,valid.val_loss,'o-',label='Validation loss')
            for col,label in [('val_precision_at_05','Precision'),('val_recall_at_05','Recall'),('val_F1','F1')]:
                axes[0,1].plot(valid.epoch,valid[col],'o-',label=label)
            axes[1,0].plot(valid.epoch,valid.val_unmatched_recall_at_05,'o-',label='Unmatched recall')
            axes[1,0].plot(valid.epoch,valid.val_recall_at_05,'o-',label='All positive recall')
            axes[1,1].plot(valid.epoch,valid.val_AP,'o-',label='AP')
            axes[1,1].plot(valid.epoch,valid.val_coverage,'o--',label='Scored / requested targets')
    if live and live.get('train',{}).get('loss') is not None:
        t=live['train'];x=live['epoch']-1+t['completed']/max(t['total'],1)
        axes[0,0].plot([x],[t['loss']],'x',label='Current training mean')
    for axis,title in zip(axes.flat,['Loss','Validation P / R / F1','Positive-group recall','AP and coverage']):
        axis.set_title(title);axis.set_xlabel('Epoch');axis.grid(alpha=.25)
        if axis.get_legend_handles_labels()[0]:axis.legend(fontsize=9)
    for axis in [axes[0,1],axes[1,0],axes[1,1]]:axis.set_ylim(0,1.03)
    fig.tight_layout();plt.close(fig);return fig


class LiveTrainingDisplay:
    def __init__(self,run_dir,config,variants):
        self.run_dir=Path(run_dir);self.config=config;self.variants=variants;self.signature=None
        self.gauge=display(HTML('진행 기록 연결 중…'),display_id=True)
        self.figure=display(HTML('첫 학습 기록 대기'),display_id=True)
        self.table=display(HTML(''),display_id=True);self.info=display(HTML(''),display_id=True)
    def refresh(self):
        active=training_is_active(self.run_dir);status=read_result_json(self.run_dir/'current_status.json',{})
        variant=status.get('variant',self.variants[0]);variant=variant if variant in self.variants else self.variants[0]
        folder=self.run_dir/'models'/variant;history=read_result_json(folder/'history.json',[])
        live=read_result_json(folder/'live_progress.json',{})
        self.gauge.update(HTML(progress_bar_html(live,self.config,variant,active)))
        signature=json.dumps([history,live.get('train'),live.get('epoch')],sort_keys=True)
        if signature!=self.signature:
            self.figure.update(learning_figure(history,variant,live))
            frame=pd.DataFrame(history[-10:]).reindex(columns=METRIC_COLUMNS)
            self.table.update(HTML('<div style="overflow:auto">'+frame.to_html(index=False,na_rep='—',float_format=lambda v:f'{v:.5f}')+'</div>'))
            self.signature=signature
        eta=read_result_json(folder/'eta.json',{})
        text='P/R/F1/AP는 공식 head에서 실제 판정된 거래 기준입니다. coverage도 함께 확인하세요. 검증하지 않은 epoch는 —로 표시합니다.'
        if eta.get('remaining_hours') is not None:text+=f"<br>현재 모델 남은 시간 약 {eta['remaining_hours']:.1f}시간 · 예상 완료(UTC): {eta['estimated_finish_utc']}"
        if not history and not active:text+='<br>수정 조건의 새 실행 기록이 없습니다. 직접 학습하려면 RUN_TRAINING=True로 설정하세요.'
        self.info.update(HTML(text));return active


def follow_training(run_dir,config,variants,refresh_seconds=2,max_updates=None):
    viewer=LiveTrainingDisplay(run_dir,config,variants);count=0
    try:
        while True:
            active=viewer.refresh();count+=1
            if not active or (max_updates is not None and count>=max_updates):break
            time.sleep(max(1,min(refresh_seconds,60)))
    except KeyboardInterrupt:print('조회 셀을 종료했습니다. 별도 프로세스의 학습 상태는 그대로입니다.')
    return viewer

## 16. 실행

학습을 시작하려면 첫 셀의 RUN_TRAINING을 True로 바꿉니다. 아래 셀은 검증된 소스와 화면에 보이는 함수의 AST가 같은지 확인한 뒤 실행합니다. 같은 RUN_NAME을 두 프로세스가 동시에 학습하지 못하도록 잠금도 적용합니다.

3개 모델의 학습이 모두 끝나면 best checkpoint를 고정하고 test 최대256배치를 한 번 평가합니다. Test 점수로 모델이나 임계값을 바꾸지 않습니다. 목표 **P≥70%, R≥65%** 달성 여부는 별도 전수 평가 없이 확정하지 않습니다.

In [ ]:
viewer = None
last_refresh = 0.0

def refresh_during_training(value):
    global last_refresh
    now = time.monotonic()
    if now - last_refresh >= REFRESH_SECONDS or value[value["phase"]]["percent"] >= 100:
        viewer.refresh()
        last_refresh = now

def refresh_after_epoch(history, variant, best_epoch, eta):
    viewer.refresh()

if RUN_TRAINING:
    if not torch.cuda.is_available():
        raise RuntimeError("전체 학습에는 CUDA GPU 커널을 사용하세요.")
    with run_lock():
        require_verification()
        prepare(CONFIG, overrides=visible_functions(globals()))
        viewer = LiveTrainingDisplay(RUN_ROOT, CONFIG, VARIANTS)
        try:
            for variant in VARIANTS:
                run_variant(CONFIG, variant, on_epoch=refresh_after_epoch,
                            on_progress=refresh_during_training)
            final_result = finish_evaluation(CONFIG, model_factory=make_model)
            display(final_result)
        except KeyboardInterrupt:
            log("notebook_interrupted", checkpoint="last saved optimizer update")
            print("중단했습니다. 같은 RUN_NAME으로 다시 실행하면 마지막 저장 지점에서 재개합니다.")
        finally:
            viewer.refresh()
elif FOLLOW_ACTIVE_TRAINING:
    viewer = follow_training(RUN_ROOT, CONFIG, VARIANTS, refresh_seconds=REFRESH_SECONDS)
else:
    print("학습은 시작하지 않았습니다. RUN_TRAINING=True이면 직접 학습합니다.")

## 17. 저장 기록 확인

지표와 중단 재개용 체크포인트는 RUN_ROOT 아래에 기록합니다. 아래 표는 이번 RUN_NAME의 기록만 읽습니다. 과거 실험의 성능을 새 조건의 결과로 표시하지 않습니다.

In [ ]:
for variant in VARIANTS:
    folder = RUN_ROOT / "models" / variant
    history = read_result_json(folder / "history.json", [])
    print(variant, "완료 epoch:", len(history))
    if history:
        display(pd.DataFrame(history).reindex(columns=METRIC_COLUMNS))
        display(learning_figure(history, variant))
print("검증 기록:", EXPERIMENT_ROOT / "verification.json")
print("변경 설명:", EXPERIMENT_ROOT / "노트북_조건수정_검증기록.html")

In [ ]:

assert not RUN_TRAINING and not FOLLOW_ACTIVE_TRAINING
assert fingerprint(visible_functions(globals())) == fingerprint(visible_functions())
assert CONFIG.graph_mode == "shared"
assert not (RUN_ROOT / "models").exists()
# All notebook-defined functions must resolve their globals, including imports.
config = replace(CONFIG, epochs=6, seed_batch=8, iterations_per_epoch=2, validation_batches=2)
assert graph_chunks([8,8], config)[1] == [8,8]
html = progress_bar_html({'epoch': 2, 'validation': {'scheduled': False}}, CONFIG, VARIANTS[0], False)
assert '이번 epoch' in html
viewer = LiveTrainingDisplay(RUN_ROOT, CONFIG, VARIANTS)
assert viewer.refresh() is False
# Cached index loading checks the new shared construction lock, without rebuilding large files.
assert build_index('train').is_dir() and build_index('val').is_dir()
print('NOTEBOOK_ALL_CELLS_AND_AST_PARITY_PASS')
